In [0]:
# ══════════════════════════════════════
# 00_CONFIG — Configurações do Projeto
# Squad 3 — Arquitetura Medalhão
# Batch Lojas Físicas
# Luiz Henrique Portácio
# ══════════════════════════════════════

# Configuração — Squad 3 (Batch Lojas Físicas)

Notebook central de configuração do pipeline. Define variáveis de
ambiente, caminhos no Data Lake (ADLS Gen2) e credenciais do SQL
Server, lendo diretamente as chaves do `.env` do projeto.

**Sobre autenticação no ADLS Gen2 (Serverless):**
No Databricks Serverless, `spark.conf.set()` para propriedades
`fs.azure.account.*` não é permitido. A autenticação OAuth via
Service Principal é passada como `.options(**adls_options)` diretamente
em cada operação `spark.read` / `spark.write`, via a função
`get_adls_options()` definida no `00_utils`.

**Regra de arquitetura:**
- Bronze e Silver gravam **exclusivamente em Delta** (ADLS, container
  `squad3`). Nenhuma das duas grava no SQL Server.
- Apenas a **Gold** grava no SQL Server, no schema `squad3`.
- Nenhum caminho ou tabela usa sufixo de versão (`_v2`, `_v3` etc.).

In [0]:
import os
from pathlib import Path


def load_env_file(env_path="../.env"):
    env_file = Path(env_path)

    if not env_file.exists():
        raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_file}")

    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()

        if not line or line.startswith("#"):
            continue

        if "=" not in line:
            continue

        key, value = line.split("=", 1)
        os.environ[key.strip()] = value.strip().strip('"').strip("'")


def get_required_env(key):
    value = os.getenv(key)

    if value is None or value.strip() == "":
        raise ValueError(f"Variável de ambiente ausente ou vazia: {key}")

    return value

In [0]:
load_env_file("../.env")

# ───────────────────────────────────────────────────────────
# Credenciais ADLS Gen2 (Service Principal)
# Nomes em MAIÚSCULO, conforme convenção do projeto
# ───────────────────────────────────────────────────────────
ADLS_CLIENT_ID            = get_required_env("ADLS_CLIENT_ID")
ADLS_TENANT_ID            = get_required_env("ADLS_TENANT_ID")
ADLS_CLIENT_SECRET        = get_required_env("ADLS_CLIENT_SECRET")
ADLS_STORAGE_ACCOUNT_NAME = get_required_env("ADLS_STORAGE_ACCOUNT_NAME")

# ───────────────────────────────────────────────────────────
# Credenciais SQL Server (usadas SOMENTE pela camada Gold)
# ───────────────────────────────────────────────────────────
SQL_HOST     = get_required_env("SQL_HOST")
SQL_DATABASE = get_required_env("SQL_DATABASE")
SQL_USERNAME = get_required_env("SQL_USERNAME")
SQL_PASSWORD = get_required_env("SQL_PASSWORD")
SQL_PORT     = "1433"


In [0]:
# ───────────────────────────────────────────────────────────
# Containers e caminhos no Data Lake
# bronze/, silver/, gold/ ficam na RAIZ do container squad3
# ───────────────────────────────────────────────────────────

# Containers
RAW_CONTAINER   = "raw"
SQUAD_CONTAINER = "squad3"

# Raiz dos containers
RAW_ROOT_PATH = (
    f"abfss://{RAW_CONTAINER}@"
    f"{ADLS_STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/"
)

SQUAD_ROOT_PATH = (
    f"abfss://{SQUAD_CONTAINER}@"
    f"{ADLS_STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/"
)

# Origem dos arquivos batch na Raw
RAW_BATCH_PATH = f"{RAW_ROOT_PATH}batch-data/"

# Caminhos base de cada camada
BRONZE_BASE_PATH = f"{SQUAD_ROOT_PATH}bronze/"
SILVER_BASE_PATH = f"{SQUAD_ROOT_PATH}silver/"
GOLD_BASE_PATH   = f"{SQUAD_ROOT_PATH}gold/"

# Tabelas de apoio da Silver (quarentena e métricas de DQ)
SILVER_QUARENTENA_BASE_PATH = f"{SQUAD_ROOT_PATH}silver/_quarentena_"
SILVER_DQ_METRICS_PATH      = f"{SQUAD_ROOT_PATH}silver/_data_quality_metrics"

# Caminhos Gold no ADLS (espelhados no SQL Server)
GOLD_LOJAS_PATH             = f"{GOLD_BASE_PATH}physical_lojas"
GOLD_ITENS_VENDA_CAIXA_PATH = f"{GOLD_BASE_PATH}physical_itens_venda_caixa"

# KPI 10 (feriado) tem granularidade loja+DIA -- diferente da tabela
# principal de itens (produto+loja+mes). Tabela Gold separada evita
# repetir/inflar o valor por produto.
GOLD_FERIADO_DIA_PATH = f"{GOLD_BASE_PATH}physical_feriado_dia"

# Tabelas Gold no SQL Server (schema squad3)
TARGET_SCHEMA                    = "squad3"
SQL_TABLE_GOLD_LOJAS             = f"{TARGET_SCHEMA}.gold_physical_lojas"
SQL_TABLE_GOLD_ITENS_VENDA_CAIXA = f"{TARGET_SCHEMA}.gold_physical_itens_venda_caixa"
SQL_TABLE_GOLD_FERIADO_DIA       = f"{TARGET_SCHEMA}.gold_physical_feriado_dia"

# Caminho do arquivo IBGE (gerado pelo 00_setup_ibge)
RAW_IBGE_MUNICIPIOS_PATH = f"{RAW_BATCH_PATH}ibge_municipios.csv"

print("Configuração carregada com sucesso.")
print(f"Storage account : {ADLS_STORAGE_ACCOUNT_NAME}")
print(f"Raw batch path  : {RAW_BATCH_PATH}")
print(f"Bronze path     : {BRONZE_BASE_PATH}")
print(f"Silver path     : {SILVER_BASE_PATH}")
print(f"Gold path       : {GOLD_BASE_PATH}")
print(f"SQL schema      : {TARGET_SCHEMA}")

# ───────────────────────────────────────────────────────────
# Paths específicos por tabela (usados pelos notebooks
# Bronze, Silver, Gold e Analysis)
# ───────────────────────────────────────────────────────────

# Raw — arquivos de origem
RAW_PHYSICAL_LOJAS_PATH             = f"{RAW_BATCH_PATH}physical_lojas.csv"
RAW_PHYSICAL_VENDAS_CAIXA_PATH      = f"{RAW_BATCH_PATH}physical_vendas_caixa.csv"
RAW_PHYSICAL_ITENS_VENDA_CAIXA_PATH = f"{RAW_BATCH_PATH}physical_itens_venda_caixa.csv"

# Bronze — Delta, append, cast string
BRONZE_LOJAS_PATH             = f"{BRONZE_BASE_PATH}physical_lojas"
BRONZE_VENDAS_CAIXA_PATH      = f"{BRONZE_BASE_PATH}physical_vendas_caixa"
BRONZE_ITENS_VENDA_CAIXA_PATH = f"{BRONZE_BASE_PATH}physical_itens_venda_caixa"

# Silver — Delta, tratado e padronizado
SILVER_LOJAS_PATH             = f"{SILVER_BASE_PATH}physical_lojas"
SILVER_ITENS_VENDA_CAIXA_PATH = f"{SILVER_BASE_PATH}physical_itens_venda_caixa"

# Silver — quarentena (PK nula/duplicada)
SILVER_QUARENTENA_LOJAS_PATH             = f"{SILVER_QUARENTENA_BASE_PATH}physical_lojas"
SILVER_QUARENTENA_ITENS_VENDA_CAIXA_PATH = f"{SILVER_QUARENTENA_BASE_PATH}physical_itens_venda_caixa"